<a href="https://colab.research.google.com/github/mAliAytekin/ai-research-notes/blob/main/t_slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## importing some important things


In [1]:
!pip install PyPDF2

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import re
import os
from PyPDF2 import PdfReader
import math
import random
import torch
import torch.nn as nn
from torch.nn import functional as F

random.seed(42)


## pdf -> txt

amelasyon iş. ai generated.

In [4]:
def pdf2txt(file_path,output_cleaned_txt_path):
  try:
      # PDF dosyasını oku
      reader = PdfReader(file_path)
      raw_text = ""
      for i, page in enumerate(reader.pages):
          try:
              # Her sayfanın metnini çıkar ve ekle
              page_text = page.extract_text()
              if page_text:
                  raw_text += page_text + "\n"
          except Exception as page_e:
              print(f"Uyarı: Sayfa {i+1} işlenirken bir hata oluştu: {page_e}. Bu sayfa atlanıyor.")
              continue # Sonraki sayfaya geç

      # Metni temizleme adımları
      # 1. Türk alfabesi dahil harf ve tire-yeni satır-harf desenini birleştir (hyphenated words)
      cleaned_text = re.sub(r'([a-zA-ZğüşıöçĞÜŞİÖÇ])-\n([a-zA-ZğüşıöçĞÜŞİÖÇ])', r'\1\2', raw_text)
      # 2. Metindeki tüm yeni satır karakterlerini boşlukla değiştir
      cleaned_text = cleaned_text.replace('\n', ' ')
      # 3. Birden çok boşluğu tek boşluğa indirge
      cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
      # 4. Sekmeleri kaldır
      cleaned_text = cleaned_text.replace('\t', ' ')
      # 5. Nokta, soru işareti veya ünlemden sonra (varsa) birden çok boşluğu tek boşluğa indirge ve ardından yeni bir satır ekle.
      cleaned_text = re.sub(r'([.?!])\s*', r'\1\n', cleaned_text)
      # Başlangıçtaki ve sondaki boşlukları temizle
      cleaned_text = cleaned_text.strip()

      # Temizlenmiş metni yeni bir dosyaya kaydet
      with open(output_cleaned_txt_path, 'w', encoding='utf-8') as f:
          f.write(cleaned_text)

      print(f"PDF içeriği başarıyla temizlendi ve {output_cleaned_txt_path} adresine kaydedildi.")

  except FileNotFoundError:
      print(f"Hata: Belirtilen yolda dosya bulunamadı: {file_path}")
  except Exception as e:
      print(f"PDF dosyasını işlerken genel bir hata oluştu: {e}")

In [5]:
file_path = '/content/drive/MyDrive/T-SLM/secme_hikayeler.pdf'
output_cleaned_txt_path = '/content/drive/MyDrive/T-SLM/hikayeler_cleaned.txt'
pdf2txt(file_path,output_cleaned_txt_path)


with open(output_cleaned_txt_path, 'r', encoding='utf-8') as f:
  docs = f.read()

print(docs[:1500])


PDF içeriği başarıyla temizlendi ve /content/drive/MyDrive/T-SLM/hikayeler_cleaned.txt adresine kaydedildi.
H İ KAYELER ÖMER SEYFETT İ N http://eskikitaplarim.
com Düzenleme: Tyrion ÖMER SEYFETT İ N 28 Şubat 1884 tarihinde Gönen'de doğdu.
Öğrenimine Gönen'de başlayan Ömer Seyfettin, Ayancık'ta ve annesiyle birlikte geldiği İstanbul'da Aksaray'daki Mekteb-i Osmaniye'ye devam etti.
Eyüp'teki Baytar Rüşdiyesi'ni bitirip asker çocuğu olduğu için Kuleli Askeri İdadi'sine yazıldı (1893).
Bir müddet sonra da Edirne Askeri İdadisi'ne naklolarak öğrenimini burada tamamladı.
Daha sonra İstanbul'da Mekteb-i Harbiye'ye gelen Ömer Seyfettin, piyade mülâzımı sânisi rütbesiyle buradan mezun oldu.
İzmir'de Teğmen (1903-1910), daha sonra da üsteğmen olarak Rumeli'de görev yaptı (1908-1910).
Askerlik'ten ayrılıp Selanik'e gelerek, Genç Kalemler Dergisi'nde yazmaya başladı.
Balkan Savaşı'nda tekrar subay olarak orduya döndü.
Yunanlılar'ın elinde bir yıl kadar esir kaldı.
Esareti sırasında da öykü yazamay

## tokenization

In [6]:
chars = sorted(list(set(docs)))
vocab_size_actual = len(chars)


stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}


def encode(s):
    return [stoi[ch] for ch in s]
def decode(l):
    return ''.join([itos[i] for i in l])

print(f"\nKelime haznesi boyutu (vocab_size): {vocab_size_actual}")
print(f"İlk 10 karakterin token karşılığı: {encode(docs[:10])}")
print(f"İlk 10 tokenin karakter karşılığı: {decode(encode(docs[:10]))}")


Kelime haznesi boyutu (vocab_size): 85
İlk 10 karakterin token karşılığı: [30, 1, 81, 1, 32, 23, 43, 27, 33, 27]
İlk 10 tokenin karakter karşılığı: H İ KAYELE


## parametreler

In [7]:
coeff = 2
batch_size = 16   # aynı anda işlenecek cümle sayısı
block_size = 64   # context window. max cümle uzunluğu
n_embd = 256       # vektör boyutu (n_head'e bölünebilir olmalı)
n_layer = 16
n_head = 8
n_ff = n_embd*4 # Genellikle n_embd'nin 4 katı olarak belirlenir.

vocab_size = len(chars) # Güncellenen vocab_size tanımı


max_iters = 10000     # kaç iterasyon olacak eğitim
eval_iters = 100

best_val_loss = 1e9

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## bağzı attention işleri

In [9]:
class Head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)

    # maskeleme
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self,x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    # print(f"Inside Head.shape {x.shape}")

    # print(f"Inside Head : {x}")

    # print(f"Inside Head k : {k}")

    # print(f"Inside Head q: {q}")

    # dikkat skorları hesaplanır Q @ K^T
    wei = q @ k.transpose(-2,-1) * C**-0.5

    # print(f"Inside Head wei1.shape: {wei.shape}")
    # print(f"Inside Head wei1: {wei}")

    # maskeleme
    wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))

    # print(f"Inside Head wei2.shape: {wei.shape}")
    # print(f"Inside Head wei2: {wei}")

    # softmax ile olasılık dönüşümü
    # hangi karakter önemli % cinsinden bulunur
    wei = F.softmax(wei,dim=-1)

    # print(f"Inside Head wei3.shape: {wei.shape}")
    # print(f"Inside Head wei3: {wei}")

    v = self.value(x)

    # print(f"Inside Head v.shape: {v.shape}")
    # print(f"Inside Head v: {v}")
    out = wei @ v

    # print(f"Inside Head out: {out}")

    return out


In [10]:
class MultiHeadAttention(nn.Module):
  def __init__(self,num_heads,head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    # çıktıyı tekrar ana boyuta n_embd dönüştür.
    self.proj = nn.Linear(head_size * num_heads,n_embd)

  def forward(self,x):
    # tüm headlerin çıktılarını birleştir
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.proj(out)
    return out

## feedforward

In [11]:
class FeedForward(nn.Module):
  def __init__(self,n_embd,n_ff):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,n_ff * n_embd),
        nn.GELU(),
        nn.Linear(n_ff * n_embd,n_embd),
        nn.Dropout(0.1)
    )

  def forward(self,x):
    return self.net(x)

## transformer block

In [12]:
class Block(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd,n_ff)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

## lang model

In [13]:
class GPTModel(nn.Module):
  def __init__(self,n_layer):
    super().__init__()

    # embedding'ler
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table = nn.Embedding(block_size,n_embd)

    # transformer blockları
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

    # layer norm ve lineer katman
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    # print("\nInside GPT")
    B,T = idx.shape
    # print(idx.shape)
    # embedding'leri hesapla
    tok_emb = self.token_embedding_table(idx)

    # print(f"\ntok_emb : {tok_emb}")

    pos_emb = self.position_embedding_table(torch.arange(T,device=device))

    # print(f"\npos_emb : {pos_emb}")


    x = tok_emb+pos_emb

    # print(f"\nx : {x}")

    # bloklardan geçiş
    x = self.blocks(x)

    # print(f"\nblocks : {x}")

    x = self.ln_f(x)

    # print(f"\nln_f : {x}")

    # logits hesapla
    logits = self.lm_head(x)
    # print(f"\nlogits.shape : {logits.shape}")

    # print(f"\nlogits : {logits}")

    loss = None
    if targets is not None:

      # eğitim sırasında cross entropy hesapla
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits,loss

  def generate(self, idx, max_new_tokens,temperature=1.0, top_k=None, top_p=None):
      # idx, (B, T) boyutunda mevcut karakterlerin indeksleridir
      for _ in range(max_new_tokens):
          # eğer mevcut metin block_size'dan uzunsa, sadece son block_size kadarını al
          # çünkü positional embedding matrisimiz sabit bir boyuta (block_size) sahip
          idx_cond = idx[:, -block_size:]

          # modeli çalıştır (forward pass)
          logits, loss = self(idx_cond)

          # sadece son zaman adımındaki (T) olasılıklara odaklan
          # logits shape: (B, T, C) -> sonuncuyu al: (B, C)
          logits = logits[:, -1, :] / temperature

          if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

          if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # eşiği geçenleri kaldır
            sorted_indices_to_remove = cumulative_probs > top_p
            # ilk indeksi koru (en az bir tane kalsın)
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[:, indices_to_remove] = -float('Inf')


          # olasılıkları almak için softmax uygula
          probs = F.softmax(logits, dim=-1) # (B, C)

          # bu olasılık dağılımından bir sonraki karakteri rastgele seç (sampling)
          idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

          # yeni karakteri mevcut diziye ekle ve döngüye devam et
          idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

      return idx

## batching

In [14]:
n = int(0.9 * len(docs))
train_data = docs[:n]
val_data = docs[n:]

# Metin verilerini tokenlere dönüştür
train_data = torch.tensor(encode(train_data), dtype=torch.long).to(device)
val_data = torch.tensor(encode(val_data), dtype=torch.long).to(device)

def get_batch(split):
    # rastgele bir başlangıç noktası
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # x -> input dizisi, y -> hedeflenen (bir sonraki) karakterler
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

## optimizer

In [15]:
model = GPTModel(n_layer).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5,weight_decay=0.1)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters, eta_min=5e-7)

## training loop

In [16]:
@torch.no_grad() # gradyan hesaplamayı kapat
def estimate_loss():
    out = {}
    model.eval() # modeli değerlendirme moduna al (Dropout vb. kapanır)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # modeli tekrar eğitim moduna al
    return out

In [17]:
it_count = 2
for iter in range(max_iters):

    # 1. veriyi al
    xb, yb = get_batch('train')
     #print(f"xb tensor:\n{xb}")
     #print("------------")
     #print(f"xb shape: {xb.shape}")
     #print("------------")
     #print(f"yb tensor:\n{yb}")
     #print("------------")

     #print("Decoded xb (first xb):")
     #print(decode(xb[0].tolist()))
     #print(decode(xb[1].tolist()))
      #print(decode(xb[2].tolist()))

     #print("\nDecoded yb (first yb):")
     #print(decode(yb[0].tolist()))
     #print(decode(yb[1].tolist()))
     #print(decode(yb[2].tolist()))
     #print("------------")


    # 2. forward pass ve loss hesaplama
    logits, loss = model(xb, yb)

    # 3. backpropagation
    optimizer.zero_grad(set_to_none=True) # Eski gradyanları temizle
    loss.backward()

    # 4. ağırlıkları güncelle
    optimizer.step()

    # 5. öğrenme hızını güncelle
    scheduler.step()


    if iter % eval_iters == 0:
      current_lr = optimizer.param_groups[0]['lr']
      losses = estimate_loss()
      print(f"Adım {iter}: train_loss {losses['train']:.4f}, val_loss {losses['val']:.4f}, LR: {current_lr:.2e}")

      if losses['val'] < best_val_loss:
          best_val_loss = losses['val']
          torch.save(model.state_dict(), 'best_model.pth')
          print(f"-- En iyi model kaydedildi (Val Loss: {best_val_loss:.4f}) --")

Adım 0: train_loss 3.5898, val_loss 3.6080, LR: 5.00e-05
-- En iyi model kaydedildi (Val Loss: 3.6080) --
Adım 100: train_loss 2.6171, val_loss 2.6538, LR: 5.00e-05
-- En iyi model kaydedildi (Val Loss: 2.6538) --
Adım 200: train_loss 2.5451, val_loss 2.5892, LR: 5.00e-05
-- En iyi model kaydedildi (Val Loss: 2.5892) --
Adım 300: train_loss 2.5068, val_loss 2.5454, LR: 4.99e-05
-- En iyi model kaydedildi (Val Loss: 2.5454) --
Adım 400: train_loss 2.4872, val_loss 2.5211, LR: 4.98e-05
-- En iyi model kaydedildi (Val Loss: 2.5211) --
Adım 500: train_loss 2.4770, val_loss 2.5107, LR: 4.97e-05
-- En iyi model kaydedildi (Val Loss: 2.5107) --
Adım 600: train_loss 2.4489, val_loss 2.5021, LR: 4.96e-05
-- En iyi model kaydedildi (Val Loss: 2.5021) --
Adım 700: train_loss 2.4421, val_loss 2.4875, LR: 4.94e-05
-- En iyi model kaydedildi (Val Loss: 2.4875) --
Adım 800: train_loss 2.4384, val_loss 2.4726, LR: 4.92e-05
-- En iyi model kaydedildi (Val Loss: 2.4726) --
Adım 900: train_loss 2.4155, v

## generation

In [75]:
start_text = "eskiden "
start_ids = encode(start_text) # Örn: [45, 28, 29, 28, 35, 1]

# shape: (1, T) -> (1, 6)
x = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

model.eval()

with torch.no_grad(): # Gradyan hesaplamaya gerek yok
  generated_indices = model.generate(x, max_new_tokens=200, temperature=0.7,top_k=100, top_p=0.8)[0].tolist()

print("---------------------------")
print("start text : ",start_text)
print(decode(generated_indices))
print("---------------------------")

---------------------------
start text :  eskiden 
eskiden bir haydi yoktu.
Karşısında kadar bakalıyordu.
Allan tek keni yatındı.
Bir yanın başına başladı.
Kenz karan aramadı.
Arsının kendirin altığın bir dayaları başka kaldı.
Ama gibi olunu takın çıktırdı.
G
---------------------------


## kaynaklar

- https://sehitkadersivriortaokulu.meb.k12.tr/meb_iys_dosyalar/34/22/736214/dosyalar/2019_09/23085831_Hikayeler_-_Omer_Seyfettin__PDFDrive.com_.pdf?CHK=58f5398efac7254796414f3fb5beef75#page=155.04

